In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

hf_path = "/home/stenheli/projects/heart-failure-detection/sandbox/ensembled_testset.csv"
df = pd.read_csv(hf_path)
echo_path = "/dataset/ecg/tabular/raw/echo_22092025.csv"
echo = pd.read_csv(echo_path)
fnr_map_path = "/dataset/ecg/tabular/raw/2025_32_echo_mapping_foedselsnr.csv"
fnr_map = pd.read_csv(fnr_map_path)

echo = echo.merge(fnr_map[["FOEDSELSNR_SINGLE", "FOEDSELSNR_DOUBLE"]], left_on="ID", right_on="FOEDSELSNR_SINGLE", how="left")
echo = echo[~echo["FOEDSELSNR_DOUBLE"].isna()]
echo = echo.rename(columns={"FOEDSELSNR_DOUBLE": "FOEDSELSNR"})
echo = echo.drop(columns=["FOEDSELSNR_SINGLE"])

N_BOOTSTRAP = 300

CACHE_DIR = "./cache/"
if not os.path.exists(CACHE_DIR):
    os.makedirs(CACHE_DIR)

In [ ]:
short_to_compacs_mapping = {
    "LVEDVOL": [
       'V_GNLVEDVOL', 'V_2DLVEDVOL', 'V_GNLVEDVOLZD', 'V_2DLVEDVOLZD'
    ],
    "CI": [
        'V_GNLVCARDIND', 'V_2DLVCARDIND'
    ],
    "CO": [
        'V_GNLVCARDOUT', 'V_DPLVCARDOUT'
    ],
    "IVT": [
        'V_MMISDTHICK'
    ],
    "PWT": [
        'V_MMLVPWDTHICK'
    ],
    "MAPSE": [
        'V_2DLVMEANMAPSE', 'V_MMLVLATMAPSE', 'V_MMLVINFSEPMAPSE'
    ],
    "LVEF": [
        'V_3DLVEF', 'V_2DLVEF', 'V_2DLVEF4CSSP', 'V_2DLVEF2CSSP', 'V_2DLVEF2C',
        'V_2DLVEF4CAL', 'V_2DLVEF2CAL', 'V_2DLVEFSBP', 'V_GNLVEF', 'V_GNLVEF_S'
    ],
    "GLS": [
        "V_2DLVLS", "V_3DLVLS", "V_2DLVLSLAX", "V_2DLVLS4C", "V_2DLVLS2C",
    ],
    "LVm": [
        'V_2DLVMASS', 'V_GNLVMASS', 'V_GNLVMASS_M', 'V_MMLVMASS',
    ],
    "RWT": [
        'V_2DLVRWTHICK', 'V_GNLVRWTHICK', 'V_GNLVRWTHICK_M', 'V_MMLVRWTHICK'
    ],
    "LAv": [
        'V_GNLAVOL','V_2DLAVOL',
    ],
    "RAv": [
        'V_GNRAVOL'
    ],
    "E": [
        'V_DPMVEPEAKVEL',
    ],
    "A": [
        'V_DPMVAPEAKVEL', 'V_DPMVAVMAXVALS'
    ],
    "LVM" : [
        "V_GNLVMASS"
    ],
    "E_DecelTime": [
        'V_DPMVEDECTIME', 'V_DPMVPHT'
    ],
    "e'": [
        "V_DPMVEPPV"
    ],
    "E/e'": [
        'V_DPMVEELATRATIO', 'V_DPMVEESEPTRATIO', 'V_DPTVEELATRATIO'
    ],
    "TAPSE": [
        'V_MMRVTAPSE'
    ],
    "PASP": [
        'V_DPPAESPRESS' 
    ],
}
echo_min_max_cutoff_mapping = {
    "CI":          (0.1  , 10.0),
    "CO":          (0.1  , 20.0),
    "IVT":         (0.1  , 5.0),
    "PWT":         (0.1  , 5.0),
    "MAPSE":       (0.1  , 3.0),
    "LVEF":        (0.0  , 100.0),
    "GLS":        (-30.0 , 0.0),
    "LVm":         (15.0 , 1000.0),
    "RWT":         (0.1  , 5.0),
    "LAv":         (0.1  , 320.0),
    "LAVi":        (0.1  , 400.0),
    "RAv":         (0.1  , 320.0),
    "E":           (0.1  , 5.0),
    "A":           (0.2  , 5.0),
    "E_DecelTime": (0.1  , 800.0),
    "e'":          (0.01 , 0.25),
    "E/e'":        (0.1  , 60.0),
    "TAPSE":       (0.1  , 4.5),
    "PASP":        (0.1  , 100.0),
    "LVEDVOL":     (10.0 , 400.0),
    "LVEDVi":      (10.0 , 250.0),
    "LVMi":        (10.0 , 300.0),
}
units_mapping = {
    "CI": "L/min/m^2",
    "CO": "L/min",
    "IVT": "cm",
    "PWT": "cm",
    "MAPSE": "cm",
    "LVEF": "%",
    "GLS": "%",
    "LVm": "g",
    "RWT": "",
    "LAv": "mL",
    "LAVi": "mL/m^2",
    "RAv": "mL",
    "E": "m/s",
    "A": "m/s",
    "E_DecelTime": "ms",
    "e'": "m/s",
    "E/e'": "",
    "TAPSE": "cm",
    "PASP": "mmHg",
    "LVEDVOL": "mL",
    "LVEDVi": "mL/m^2",
    "LVMi": "g/m^2",
}




In [ ]:
df_pids = df["FOEDSELSNR"]
echo_pids = echo["FOEDSELSNR"]
overlap = np.intersect1d(df_pids, echo_pids)
df = df[df["FOEDSELSNR"].isin(overlap)]
echo = echo[echo["FOEDSELSNR"].isin(overlap)]

print(f"Number of patients in df before merge: {df['FOEDSELSNR'].nunique()}")
print(f"Number of patients in echo before merge: {echo['FOEDSELSNR'].nunique()}")
print(f"Number of patients after merge: {df['FOEDSELSNR'].nunique()}")

In [ ]:
def first_available_column(df: pd.DataFrame, candidates):
    """Return the first column from candidates that exists in df, else None."""
    for c in candidates:
        if c in df.columns:
            return c
    return None

def coalesce_series(df: pd.DataFrame, candidates):
    """
    Return a single Series coalescing across candidates (first non-null per row),
    using only columns present in df. Returns (series, cols_used).
    """
    avail = [c for c in candidates if c in df.columns]
    if not avail:
        return pd.Series(index=df.index, dtype=float), []
    s = df[avail].bfill(axis=1).iloc[:, 0]
    return s, avail

# Build a tidy DataFrame with ONE representative Series per group
group_series = {}
chosen_columns = {}  # track which columns were used per group
for grp, candidates in short_to_compacs_mapping.items():
    s, used = coalesce_series(echo, candidates)
    group_series[grp] = s
    chosen_columns[grp] = used

# add back timestamp and FOEDSELSDATO from the original dataframe
meta_cols = ["SeriesDate", "FOEDSELSNR"]  # include patient ID too if useful
for col in meta_cols:
    if col in echo.columns:
        group_series[col] = echo[col]

echo_hf_grouped = pd.DataFrame(group_series, index=echo.index)

# find the number of unique patients that has measured GLS
num_gls = echo_hf_grouped[~echo_hf_grouped["GLS"].isna()]["FOEDSELSNR"].nunique()
print(f"Number of unique patients with GLS measured: {num_gls}")

In [ ]:
# for E and A, use the thresholds to set some values to nan, then, for plot E/A
import seaborn as sns
E_lower, E_upper = echo_min_max_cutoff_mapping["E"]
A_lower, A_upper = echo_min_max_cutoff_mapping["A"]
E = echo_hf_grouped["E"].copy()
A = echo_hf_grouped["A"].copy()
E[(E < E_lower) | (E > E_upper)] = np.nan
A[(A < A_lower) | (A > A_upper)] = np.nan
EA_ratio = E / A
echo_hf_grouped["E/A"] = EA_ratio

# print the number of values
E_count = E.notna().sum()
A_count = A.notna().sum()
print(f"E: {E_count} values after thresholding")
print(f"A: {A_count} values after thresholding")

# plot a histogram of each
fig, ax = plt.subplots(figsize=(8, 6))
sns.kdeplot(E.dropna(), ax=ax, label=f"E (n={E_count})", lw=2, bw_adjust=0.2)
sns.kdeplot(A.dropna(), ax=ax, label=f"A (n={A_count})", lw=2, bw_adjust=0.2)
ax.set_title("E and A after thresholding")
ax.legend()
ax.set_xlabel("Value")
ax.set_ylabel("Density")
plt.tight_layout()
plt.show()

EA_count = EA_ratio.notna().sum()
print(f"E/A Ratio: {EA_count} values after thresholding")

# show
fig, ax = plt.subplots(figsize=(8, 6))
sns.kdeplot(EA_ratio.dropna(), ax=ax, label=f"E/A Ratio (n={EA_count})", lw=2, bw_adjust=0.2)
ax.set_title ("E/A Ratio after thresholding")
ax.legend()
ax.set_xlabel("Value")
ax.set_ylabel("Density")
plt.tight_layout()
plt.show()


In [ ]:
card_output = echo_hf_grouped["CO"]
card_index = echo_hf_grouped["CI"]
ratio = card_output / card_index
echo_hf_grouped["BSA"] = ratio
echo_hf_grouped["LVEDVi"] = echo_hf_grouped["LVEDVOL"] / echo_hf_grouped["BSA"]
# la volume thresholds
echo_hf_grouped["LAv"] = echo_hf_grouped["LAv"].where(
    (echo_hf_grouped["LAv"] >= echo_min_max_cutoff_mapping["LAv"][0]) &
    (echo_hf_grouped["LAv"] <= echo_min_max_cutoff_mapping["LAv"][1])
)
LAVi = echo_hf_grouped["LAv"] / echo_hf_grouped["BSA"]
echo_hf_grouped["LAVi"] = LAVi

LVM = echo_hf_grouped["LVM"]
echo_hf_grouped["LVMi"] = LVM / echo_hf_grouped["BSA"]

volume = echo_hf_grouped["LVEDVOL"]
fig, ax = plt.subplots(figsize=(8, 6))
sns.kdeplot(volume.dropna(), ax=ax, label=f"LVEDVOL (n={volume.notna().sum()})", lw=2, bw_adjust=0.2)
ax.set_title("LVEDVOL")
ax.legend()
ax.set_xlabel("Value")
ax.set_ylabel("Density")
plt.tight_layout()
plt.show()


fig, ax = plt.subplots(figsize=(8, 6))
sns.kdeplot(echo_hf_grouped["LVEDVi"].dropna(), ax=ax, label=f"LVEDVi (n={echo_hf_grouped['LVEDVi'].notna().sum()})", lw=2, bw_adjust=0.2)
ax.set_title("LVEDVi")
ax.legend()
ax.set_xlabel("Value")
ax.set_ylabel("Density")
plt.tight_layout()
plt.show()

LAv = echo_hf_grouped["LAv"]
fig, ax = plt.subplots(figsize=(8, 6))
sns.kdeplot(LAv.dropna(), ax=ax, label=f"LAv (n={LAv.notna().sum()})", lw=2, bw_adjust=0.2)
ax.set_title("LAv")
ax.legend()
ax.set_xlabel("Value")
ax.set_ylabel("Density")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8, 6))
sns.kdeplot(LAVi.dropna(), ax=ax, label=f"LAVi (n={LAVi.notna().sum()})", lw=2, bw_adjust=0.2)
ax.set_title("LAVi")
ax.legend()
ax.set_xlabel("Value")
ax.set_ylabel("Density")
plt.tight_layout()
plt.show()

# drop LA volume to avoid redundancy, and LVEDVOL to avoid collinearity with indexed version
echo_hf_grouped = echo_hf_grouped.drop(columns=["LAv", "LVEDVOL", "CO"])

In [ ]:
# use the thresholds to nan
for name in echo_min_max_cutoff_mapping.keys():
    if name not in echo_hf_grouped.columns:
        continue
    # count occurences before
    count_before = echo_hf_grouped[name].notna().sum()
    lower, upper = echo_min_max_cutoff_mapping.get(name, (None, None))
    if lower is not None:
        echo_hf_grouped[name][echo_hf_grouped[name] < lower] = np.nan
    if upper is not None:
        echo_hf_grouped[name][echo_hf_grouped[name] > upper] = np.nan
    # count occurences after
    count_after = echo_hf_grouped[name].notna().sum()
    print(f"{name}: {count_before} -> {count_after}")

    

In [ ]:
# echo_hf_grouped.columns
# len(echo_hf_grouped)
echo_hf_grouped
# dropna on SeriesDate
echo_hf_grouped = echo_hf_grouped.dropna(subset=["SeriesDate"])
# # save to csv
# echo_hf_grouped.to_csv("/home/stenheli/projects/heart-failure-phenotyping/data/hf/echo_hf_grouped_.csv", index=False)

In [ ]:
# 1) Define a 2×5 grid layout (10 groups total)
grid_layout = [
    ["LVEF", "GLS", "E/e'", "A", "CI", "e'", "LAVi", "LVMi"],
    ["PASP", "E/A", "MAPSE", "IVT", "PWT", "LVEDVi", "TAPSE"],
]
columns = ["age", "sex"] + grid_layout[0] + grid_layout[1]

# 2) Helper to clamp a group's series with your quantile rules
def clamp_group(series: pd.Series, group_name: str) -> pd.Series:
    s = series.dropna()
    return s

# 3) Build a tidy long DataFrame with values + grid positions (row, col)
rows = []
pos_map = {}  # (group -> (row, col)) just for convenience
for r, row_groups in enumerate(grid_layout):
    for c, grp in enumerate(row_groups):
        pos_map[grp] = (r, c)
        if grp not in echo_hf_grouped.columns:
            continue
        s = clamp_group(echo_hf_grouped[grp], grp)
        if s.empty:
            continue
        tmp = pd.DataFrame({
            "group": grp,
            "value": s,
            "row": r,
            "col": c
        }, index=s.index)
        rows.append(tmp)

grid_values = pd.concat(rows, axis=0, ignore_index=False).reset_index(names="index_id")

# 4) Optional: make a wide DataFrame in the grid order (columns exactly 10 in 2×5 order)
ordered_groups = [g for row in grid_layout for g in row]
grid_wide = echo_hf_grouped[ordered_groups].copy()

# apply clamping to each column (same rule set)
# for g in ordered_groups:
#     if g in grid_wide.columns:
#         s = grid_wide[g].dropna()
#         if not s.empty:
#             if g == "E/A":
#                 lo = s.quantile(0.01); hi = s.quantile(0.96)
#             else:
#                 lo = s.quantile(0.01); hi = s.quantile(0.995)
#             grid_wide[g] = grid_wide[g].clip(lower=lo, upper=hi)

# 5) Example plotting: 2×5 subplots using the new grid positions
fig, axes = plt.subplots(2, 9, figsize=(22, 7))
for grp in ordered_groups:
    r, c = pos_map[grp]
    s = grid_wide[grp].dropna()
    if s.empty:
        axes[r, c].set_visible(False)
        continue
    axes[r, c].hist(s.values, bins=20)
    meta = ""  # show which primary column fed this group if you want
    if grp in chosen_columns and chosen_columns[grp]:
        meta = f"\nprimary: {chosen_columns[grp][0]}"
    axes[r, c].set_title(f"{grp}{meta}")
    axes[r, c].set_xlabel(grp)
    axes[r, c].set_ylabel("Count")

plt.tight_layout()
plt.show()

patient_id = "FOEDSELSNR"

# unique_counts = {}
# for grp in echo_hf_grouped.columns:
#     mask = echo_hf_grouped[grp].notna()
#     unique_counts[grp] = echo.loc[mask, patient_id].nunique()

# print("Unique patients with non-NaN values per HF diagnostic group:")
# for grp, count in unique_counts.items():
#     print(f"{grp:20s}: {count}")

# apply the same clipping of values to the original echo_hf_grouped DataFrame
for grp in echo_hf_grouped.columns:
    if grp in grid_wide.columns:
        echo_hf_grouped[grp] = grid_wide[grp]



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# --- 0) Parse times & keep first echo per patient ---
echo_hf_grouped["SeriesDate"] = pd.to_datetime(echo_hf_grouped["SeriesDate"], errors="coerce")
df["time"] = pd.to_datetime(df["time"], errors="coerce")

# forward and back fill values starting with "V_" in echo_hf_grouped
echo_hf_grouped = echo_hf_grouped.sort_values(by=["FOEDSELSNR", "SeriesDate"])
echo_hf_grouped.update(echo_hf_grouped.groupby("FOEDSELSNR").bfill())


print(len(echo_hf_grouped))
echo_first = (
    echo_hf_grouped
    .dropna(subset=["FOEDSELSNR", "SeriesDate"])
    .sort_values(["FOEDSELSNR", "SeriesDate"])
    .drop_duplicates(subset=["FOEDSELSNR"], keep="first")
    .reset_index(drop=True)
)
print(len(echo_first))


df_times = df.dropna(subset=["FOEDSELSNR", "time"])[["FOEDSELSNR", "time"]].copy()

# --- 1) Build candidate pairs (cartesian per patient) ---
# One row per (patient’s first echo, each ECG time for that patient)
cand = echo_first[["FOEDSELSNR", "SeriesDate"]].merge(
    df_times, on="FOEDSELSNR", how="left"
)

# If a patient has no ECG rows, `time` will be NaT
cand["abs_diff"] = (cand["time"] - cand["SeriesDate"]).abs()

# --- 2) Pick the nearest ECG per patient (argmin of abs_diff) ---
# Keep at least one row per patient; idxmin ignores all-NaT by returning NaN, handle separately
idx = (
    cand
    .dropna(subset=["time"])                 # only patients with at least one ECG
    .groupby("FOEDSELSNR")["abs_diff"]
    .idxmin()
)

nearest = cand.loc[idx].copy()

# Patients with no ECG at all:
no_ecg_patients = set(echo_first["FOEDSELSNR"]) - set(nearest["FOEDSELSNR"])
if no_ecg_patients:
    # keep their echo; time will be NaT
    nearest = pd.concat([
        nearest,
        echo_first.loc[echo_first["FOEDSELSNR"].isin(no_ecg_patients), ["FOEDSELSNR", "SeriesDate"]]
        .assign(time=pd.NaT, abs_diff=pd.NaT)
    ], ignore_index=True)

# --- 3) Attach echo_first columns back (if you want all echo covariates) ---
matched = nearest.merge(
    echo_first, on=["FOEDSELSNR", "SeriesDate"], how="left", suffixes=("", "_echo")
)

# --- 4) Compute signed/absolute delta (ECG - Echo) ---
matched["delta"] = matched["time"] - matched["SeriesDate"]
matched["delta_days"] = matched["delta"].dt.total_seconds() / (24*3600)
matched["abs_delta_days"] = matched["delta_days"].abs()

print(f"First-echo patients: {len(echo_first)}")
print(f"Matched with at least one ECG: {matched['time'].notna().sum()}")
print(f"No ECG found: {matched['time'].isna().sum()}")


In [ ]:
# echo_hf_grouped = echo_hf_grouped.drop_duplicates(subset=["FOEDSELSNR"], keep="first")

# echo_hf_grouped
# # save 2022 and later
# echo_hf_grouped = echo_hf_grouped[echo_hf_grouped["SeriesDate"].dt.year >= 2023]
# # put date first, then FOEDSELSNR
# echo_hf_grouped = echo_hf_grouped[["SeriesDate", "FOEDSELSNR"] + [col for col in echo_hf_grouped.columns if col not in ["SeriesDate", "FOEDSELSNR"]]]
# echo_hf_grouped.to_csv("echo_hf_grouped_2023onwards.csv", index=False)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

MAX_DAYS = 3
td_limit = pd.Timedelta(days=MAX_DAYS)

# --- 1) prepare ECG dataframe with extra cols ---
# keep only relevant ECG cols
df_times = df.dropna(subset=["FOEDSELSNR", "time"])[
    ["FOEDSELSNR", "time", "ensemble", "age", "sex", "hf_icd_flag_ever"]
].copy()

# --- 2) build candidate pairs per patient ---
cand = echo_first[["FOEDSELSNR", "SeriesDate"]].merge(
    df_times, on="FOEDSELSNR", how="left"
)
cand["abs_diff"] = (cand["time"] - cand["SeriesDate"]).abs()

# --- 3) nearest ECG per patient (argmin per FOEDSELSNR) ---
idx = (
    cand.dropna(subset=["time"])
    .groupby("FOEDSELSNR")["abs_diff"]
    .idxmin()
)

nearest = cand.loc[idx].copy()

# patients without any ECG → keep echo only (ECG cols will be NaN)
no_ecg_patients = set(echo_first["FOEDSELSNR"]) - set(nearest["FOEDSELSNR"])
if no_ecg_patients:
    nearest = pd.concat([
        nearest,
        echo_first.loc[
            echo_first["FOEDSELSNR"].isin(no_ecg_patients),
            ["FOEDSELSNR", "SeriesDate"]
        ].assign(time=pd.NaT, abs_diff=pd.NaT, ensemble=np.nan, age=np.nan, sex=np.nan)
    ], ignore_index=True)

# --- 4) attach echo covariates (other grouped HF values) ---
matched = nearest.merge(
    echo_first, on=["FOEDSELSNR", "SeriesDate"], how="left", suffixes=("", "_echo")
)

# --- 5) compute time deltas ---
matched["delta"] = matched["time"] - matched["SeriesDate"]
matched["delta_days"] = matched["delta"].dt.total_seconds() / (24*3600)
matched["abs_delta_days"] = matched["delta_days"].abs()

# --- 6) apply timedelta limit (drop echoes with no close ECG) ---
keep_mask = matched["delta"].notna() & (matched["delta"].abs() <= td_limit)
kept = matched.loc[keep_mask].copy()
kept["age"] = kept["age"].str.extract(r"(\d+)").astype(int)
# drop age < 18 and age > 100 and print then umber of dropped patients
age_mask = (kept["age"] >= 18) & (kept["age"] <= 100)
dropped_age = len(kept) - age_mask.sum()
if dropped_age > 0:
    print(f"Dropping {dropped_age} patients due to age < 18 or > 100")
kept = kept.loc[age_mask]
kept["sex"] = kept["sex"].replace({"M": 1.0, "F": 0.0, "O": 0.5})

print(f"Total first-echo patients: {len(matched)}")
print(f"Kept within ±{MAX_DAYS} days: {len(kept)}")
print(f"Dropped: {len(matched) - len(kept)}")

# --- 7) histogram ---
s = kept["delta_days"]
if not s.empty:
    # plt.figure(figsize=(6,4))
    fig, ax = plt.subplots(figsize=(4,3))
    ax.hist(-s, bins=int(MAX_DAYS*24+1), edgecolor="C0")
    ax.set_xlabel(f"Time from ECG to echocardiography (days)")
    ax.set_ylabel("Patients")
    fig.tight_layout()
    ax.set_xticks(np.arange(-int(MAX_DAYS), int(MAX_DAYS)+1, 1))
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_yticks([1000, 2000, 3000], labels=["1k", "2k", "3k"])
    plt.savefig("figures/png/ecg_echo_time_difference_histogram.png", dpi=300)
    plt.savefig("figures/pgf/ecg_echo_time_difference_histogram.pgf", bbox_inches='tight')
    plt.show()

    # print the mean absolute differnece in s
    mean_abs_diff = s.abs().mean()
    print(f"Mean absolute time difference between ECG and Echo: {mean_abs_diff*24:.2f} hours")
    # count how often before vs affter
    before_count = (s < 0).sum()
    after_count = (s > 0).sum()
    print(f"ECG before Echo: {before_count} times")
    print(f"ECG after Echo: {after_count} times")

    # # same plot but absolute differences
    # fig, ax = plt.subplots(figsize=(6,4))
    # ax.hist(s.abs(), bins=int(MAX_DAYS*24+1))
    # ax.set_xlabel(f"Matches |ECG - Echo| (days)")
    # ax.set_ylabel("Count")
    # ax.set_xticks(np.arange(0, MAX_DAYS+1, 1))
    # ax.set_xlim(0, MAX_DAYS)
    # fig.tight_layout()
    # # top right and spines visible
    # ax.spines['top'].set_visible(False)
    # ax.spines['left'].set_visible(False)
    # ax.spines['right'].set_visible(False)
    # plt.show()
    # # same but cumsum
    # sorted_abs = np.sort(s.abs())
    # cumsum = np.arange(1, len(sorted_abs)+1) / len(sorted_abs)
    # plt.figure(figsize=(6,4))
    # plt.plot(sorted_abs, cumsum)
    # plt.xlabel(f"|ECG - Echo| (days)")
    # plt.ylabel("Cumulative Count")
    # plt.title("Cumulative Count of Nearest ECG vs First Echo per Patient")
    # plt.tight_layout()
    # plt.axhline(0.8, color='red', linestyle='--')
    # plt.axhline(0.9, color='green', linestyle='--')
    # plt.show()


else:
    print("No pairs within the specified time window.")


In [ ]:
cache_file_name = "diagnoses.csv"
if not os.path.exists(os.path.join(CACHE_DIR, cache_file_name)):
    diag = pd.read_csv("/dataset/ecg/tabular/raw/2025_32_Diagnoser.csv", dtype="str")
    demo = pd.read_csv("/dataset/ecg/tabular/raw/2025_32_Demorgafi.csv", dtype="str")
    demo_pids = demo["PERSONID"].unique().tolist()
    diag_pids = diag["PERSONID"].unique().tolist()
    common_pids = set(demo_pids).intersection(set(diag_pids))
    demo_foedselsnrs = demo["FOEDSELSNR"].unique().tolist()
    echo_foedselsnrs = kept["FOEDSELSNR"].unique().tolist()
    common_foedselsnrs = set(demo_foedselsnrs).intersection(set(echo_foedselsnrs))
    subdemo = demo[demo["FOEDSELSNR"].isin(common_foedselsnrs)].copy()
    subdiag = diag[diag["PERSONID"].isin(subdemo["PERSONID"].unique())].copy()
    # add the FOEDSELSNR to subdiag by merging on PERSONID
    subdiag = subdiag.merge(subdemo[["PERSONID", "FOEDSELSNR"]], on="PERSONID", how="left")


    # endow subdiag with ecg_date, taken from the "kept" dataframe
    subdiag = subdiag.merge(
        kept[["FOEDSELSNR", "time"]].drop_duplicates(),
        on="FOEDSELSNR",
        how="left"
    )

    subdiag.to_csv(os.path.join(CACHE_DIR, cache_file_name), index=False)
else:
    subdiag = pd.read_csv(os.path.join(CACHE_DIR, cache_file_name), dtype="str")
subdiag["DIAGNOSETID"] = pd.to_datetime(subdiag["DIAGNOSETID"])
subdiag["time"] = pd.to_datetime(subdiag["time"])

# from "df", add the age and sex columns to subdiag by merging on PERSONID
df["age"] = pd.to_numeric(df["age"].str.replace("Y", ""))
df["sex"] = df["sex"].str.lower()
subdiag = subdiag.merge(df[["PERSONID", "age", "sex"]], on="PERSONID", how="left")

subdiag["IS_BASELINE"] = (subdiag["DIAGNOSETID"] <= subdiag["time"] + pd.Timedelta(days=-1)).astype(int)

COMORBIDITY_REGEX = {
    "Hypertension": [r"^I1[0-5]"],
    "COPD": [r"^J44"],
    "Ischaemic heart disease": [r"^I2[0-5]", r"^I251", r"^I200", r"^I209"],
    "Atrial fibrillation": [r"^I48", r"^I481", r"^I482", r"^I483"],
    "Heart failure": [r"^I50", r"^I42", r"^I110"],
    "Myocardial infarction": [r"^I21", r"^I22", r"^I252"],
}

COMORBIDITY_MAP = {
    "Hypertension": ["I10-I15"],
    "COPD": ["J44"],
    "Ischaemic heart disease": ["I20-I25"],
    "Atrial fibrillation": ["I48"],
    "Heart failure": ["I50", "I42", "I110"],
    "Myocardial infarction": ["I21", "I22", "I252"],
}

In [ ]:
for comorbidity, patterns in COMORBIDITY_REGEX.items():
    kept[comorbidity] = 0
    mask = subdiag["KODE"].str.match("|".join(patterns))
    affected_foedselsnrs = subdiag.loc[mask, "FOEDSELSNR"].unique().tolist()
    kept.loc[kept["FOEDSELSNR"].isin(affected_foedselsnrs), comorbidity] = 1

    print(f"Comorbidity '{comorbidity}': {kept[comorbidity].sum()} patients")

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

plot_cols = [c for c in columns if c not in ("sex", "age")]

kept_filtered = kept.copy()  # .query("LVEF > 50")

# --- bootstrap settings ---
rng = np.random.default_rng(0)  # for reproducibility

# --- compute Spearman correlations + 95% CI (bootstrap) ---
corr_results = []
for col in [c for c in plot_cols if c != "sex"]:
    if col == "FOEDSELSNR":
        continue

    x = kept_filtered[col].to_numpy()
    y = kept_filtered["ensemble"].to_numpy()

    # drop NaNs for the pair
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]

    if x.size < 3:
        # not enough data to compute correlation reliably
        continue

    # point estimate
    corr, _ = spearmanr(x, y)

    # bootstrap CI
    boot_corrs = []
    n = x.size
    for _ in range(N_BOOTSTRAP):
        idx = rng.integers(0, n, size=n)
        bx = x[idx]
        by = y[idx]
        bcorr, _ = spearmanr(bx, by)
        if np.isfinite(bcorr):
            boot_corrs.append(bcorr)

    boot_corrs = np.asarray(boot_corrs)
    ci_low = np.percentile(boot_corrs, 2.5)
    ci_high = np.percentile(boot_corrs, 97.5)

    corr_results.append((col, corr, ci_low, ci_high))

corr_df = pd.DataFrame(
    corr_results,
    columns=["parameter", "spearman_corr", "ci_low", "ci_high"],
)

# sort plot_cols by absolute correlation (using point estimate)
plot_cols = sorted(
    plot_cols,
    key=lambda c: abs(
        corr_df.loc[corr_df["parameter"] == c, "spearman_corr"].values[0]
    ) if (corr_df["parameter"] == c).any() else -np.inf,
    reverse=True,
)

# --- sort & format ---
corr_df = (
    corr_df
    .assign(abs_corr=lambda d: d["spearman_corr"].abs())
    .sort_values("abs_corr", ascending=False)
    .drop(columns=["abs_corr"])
)

# --- build LaTeX body rows manually ---
rows = []
for _, row in corr_df.iterrows():
    rows.append(
        "      "
        f"{row['parameter']} & "
        f"\\num{{{row['spearman_corr']:.2f}}} & "
        f"\\num{{{row['ci_low']:.2f}}} & "
        f"\\num{{{row['ci_high']:.2f}}} \\\\"
    )
body = "\n".join(rows) + "\n"

tabular = (
    "   \\begin{tabular}{lrrr}\n"
    "      \\toprule\n"
    "      Parameter & Spearman $\\rho$ & "
    " \\multicolumn{2}{c}{95\\% CI} \\\\\n"
    "      & & Low & High \\\\\n"
    "      \\midrule\n"
    f"{body}"
    "      \\bottomrule\n"
    "   \\end{tabular}\n"
)


latex = (
    "\\begin{table}\n"
    "   \\centering\n"
    "   \\begin{threeparttable}\n"
    "   \\caption{Spearman rank correlation between parameters and ensemble prediction.}\n"
    "   \\label{tab:spearman_corr}\n"
    f"{tabular.replace('_', ' ')}"
    "   \\begin{tablenotes}\\footnotesize\n"
    "   \\item Higher absolute $\\rho$ indicates stronger monotonic association. "
    "Values are 95\\% bootstrap confidence intervals (\\num{10000} resamples).\n"
    "   \\end{tablenotes}\n"
    "   \\end{threeparttable}\n"
    "\\end{table}\n"
)

print(latex)

with open("tables/spearman_table.tex", "w") as f:
    f.write(latex)


In [ ]:
from scipy.stats import spearmanr, norm
import numpy as np
import pandas as pd


# ---------- helper to compute correlations for a given dataframe ----------
def compute_corr_for_df(df, param_cols, target_col="ensemble"):
    out = []
    for col in param_cols:
        if col in ("sex", "FOEDSELSNR"):
            continue
        corr, _ = spearmanr(df[col], df[target_col], nan_policy="omit")
        out.append((col, corr))
    return pd.DataFrame(out, columns=["parameter", "rho"])

# ---------- overall correlations (used for ordering + reporting if needed) ----------
overall_corr_df = compute_corr_for_df(kept_filtered, plot_cols)

# sort by |ρ| overall
overall_corr_df = (
    overall_corr_df
    .assign(abs_corr=lambda d: d["rho"].abs())
    .sort_values("abs_corr", ascending=False)
    .drop(columns=["abs_corr"])
)

# we'll keep the parameter order fixed from here on
param_order = overall_corr_df["parameter"].tolist()
pretty_names = overall_corr_df["parameter"].tolist()

# ---------- LVEF subgroup correlations ----------
lvef_groups = {
    r"$\leq$ 40\%": kept_filtered["LVEF"] <= 40,
    r"41--49\%": (kept_filtered["LVEF"] > 40) & (kept_filtered["LVEF"] < 50),
    r"$\geq$ 50\%": kept_filtered["LVEF"] >= 50,
}

lvef_corrs = {}
for label, mask in lvef_groups.items():
    df_sub = kept_filtered.loc[mask]
    # compute on the same param_order
    rhos = []
    for col in param_order:
        corr, _ = spearmanr(df_sub[col], df_sub["ensemble"], nan_policy="omit")
        rhos.append(corr)
    lvef_corrs[label] = rhos

# ---------- sex subgroup correlations ----------
# detect unique non-missing sex codes
sex_values = [v for v in kept_filtered["sex"].dropna().unique()]

# map sex values to labels (adjust if you know which is male/female)
sex_label_map = {val: f"sex = {val}" for val in sex_values}

sex_corrs = {}
for val in sex_values:
    label = sex_label_map[val]
    df_sub = kept_filtered.loc[kept_filtered["sex"] == val]
    rhos = []
    for col in param_order:
        corr, _ = spearmanr(df_sub[col], df_sub["ensemble"], nan_policy="omit")
        rhos.append(corr)
    sex_corrs[label] = rhos

# ---------- formatting helpers ----------
def fmt_rho(x):
    if pd.isna(x):
        return ""
    return f"{x:.2f}"

def fisher_z(r):
    r = np.clip(r, -0.999999, 0.999999)
    return np.arctanh(r)

def pval_diff_spearman(r1, n1, r2, n2):
    """Two-sided p-value for difference in independent correlations (not used after switching to permutation tests)."""
    if np.isnan(r1) or np.isnan(r2) or n1 < 4 or n2 < 4:
        return np.nan
    z1, z2 = fisher_z(r1), fisher_z(r2)
    z_diff = (z1 - z2) / np.sqrt(1/(n1 - 3) + 1/(n2 - 3))
    p = 2 * (1 - norm.cdf(abs(z_diff)))
    return p

# ---------- permutation test for difference in male vs female correlations ----------
def perm_test_diff_spearman_overall(df, param, target="ensemble", B=1000, seed=123):
    """
    Two-sided permutation test p-value for difference in Spearman correlations
    between males (sex == 1.0) and females (sex == 0.0), for a given parameter.
    """
    rng = np.random.default_rng(seed)

    sub = df[[param, target, "sex"]].dropna()

    m = sub[sub["sex"] == 1.0]
    f = sub[sub["sex"] == 0.0]

    if len(m) < 4 or len(f) < 4:
        return np.nan

    # observed difference
    rho_m, _ = spearmanr(m[param], m[target])
    rho_f, _ = spearmanr(f[param], f[target])
    obs_diff = rho_m - rho_f

    diffs = np.empty(B)

    for b in range(B):
        permuted = sub.copy()
        permuted["sex"] = rng.permutation(permuted["sex"].values)

        m_p = permuted[permuted["sex"] == 1.0]
        f_p = permuted[permuted["sex"] == 0.0]

        # if a permutation accidentally gives too few of one sex, skip by setting diff to 0
        if len(m_p) < 4 or len(f_p) < 4:
            diffs[b] = 0.0
            continue

        rho_m_p, _ = spearmanr(m_p[param], m_p[target])
        rho_f_p, _ = spearmanr(f_p[param], f_p[target])
        diffs[b] = rho_m_p - rho_f_p

    p = np.mean(np.abs(diffs) >= np.abs(obs_diff))
    return p

# Compute sample sizes for sex groups
n_male = kept_filtered.loc[kept_filtered["sex"] == 1.0].shape[0]
n_female = kept_filtered.loc[kept_filtered["sex"] == 0.0].shape[0]

# Existing correlations
sex_cols = list(sex_corrs.keys())
n_sex = len(sex_cols)
lvef_cols = list(lvef_groups.keys())
n_lvef = len(lvef_cols)

# ===== NEW: first collect all rhos to find global max |rho| =====
row_data = []
all_abs_rhos = []

for pname, param in zip(pretty_names, param_order):
    # LVEF correlations for this parameter
    rhos_lvef = [
        spearmanr(
            kept_filtered.loc[lvef_groups[c], param],
            kept_filtered.loc[lvef_groups[c], "ensemble"],
            nan_policy="omit"
        )[0]
        for c in lvef_cols
    ]
    # Sex correlations
    rho_male = spearmanr(
        kept_filtered.loc[kept_filtered["sex"] == 1.0, param],
        kept_filtered.loc[kept_filtered["sex"] == 1.0, "ensemble"],
        nan_policy="omit"
    )[0]
    rho_female = spearmanr(
        kept_filtered.loc[kept_filtered["sex"] == 0.0, param],
        kept_filtered.loc[kept_filtered["sex"] == 0.0, "ensemble"],
        nan_policy="omit"
    )[0]

    # permutation p-value (male vs female)
    pval = perm_test_diff_spearman_overall(kept_filtered, param, B=N_BOOTSTRAP)

    # store row-wise
    row_data.append(
        {
            "pname": pname,
            "rhos_lvef": rhos_lvef,
            "rho_male": rho_male,
            "rho_female": rho_female,
            "pval": pval,
        }
    )

    # collect absolute rhos (skip NaNs)
    for r in rhos_lvef + [rho_male, rho_female]:
        if not np.isnan(r):
            all_abs_rhos.append(abs(r))

max_abs_rho = max(all_abs_rhos) if all_abs_rhos else 0.0

# map |rho| to green!xx with:
#   xx = 50 for max |rho|
#   xx = 0 for |rho| <= 0.4 * max |rho|
#   linearly in between otherwise
def fmt_rho_colored(rho, max_abs):
    if pd.isna(rho) or max_abs == 0:
        return ""
    abs_r = abs(rho)
    half = 0.4 * max_abs
    if abs_r <= half:
        level = 0
    elif abs_r >= max_abs:
        level = 50
    else:
        level = int(round(50 * (abs_r - half) / (max_abs - half)))
    # wrap the number in \num
    return f"\\cellcolor{{green!{level}}} \\num{{{rho:.2f}}}"

# ===== build LaTeX rows using colored cells =====
rows = []
for rd in row_data:
    lvef_cells = [fmt_rho_colored(r, max_abs_rho) for r in rd["rhos_lvef"]]
    male_cell = fmt_rho_colored(rd["rho_male"], max_abs_rho)
    female_cell = fmt_rho_colored(rd["rho_female"], max_abs_rho)
    pval = rd["pval"]

    if pd.isna(pval):
        p_str = ""
    else:
        if pval < 0.001:
            # cap at <0.001, numeric part wrapped in \num
            p_str = r"<\num{0.001}"
        else:
            p_str = rf"\num{{{pval:.3f}}}"

    row = "      " + " & ".join(
        [rd["pname"]] + lvef_cells + [male_cell, female_cell, p_str]
    ) + r" \\"
    rows.append(row)

# ----- table layout -----
col_spec_all = "l" + "r" * (n_lvef + n_sex + 1)

header_top = (
    "      & "
    + f"\\multicolumn{{{n_lvef}}}{{c}}{{LVEF subgroup}} & "
    + f"\\multicolumn{{{n_sex + 1}}}{{c}}{{Sex}} \\\\"
)

header_mid = (
    "      Parameter & "
    + " & ".join(lvef_cols + ["Male", "Female", "p"]) + r" \\"
)

cmidrule_lvef = f"      \\cmidrule(lr){{2-{1 + n_lvef}}}"
cmidrule_sex = f"      \\cmidrule(lr){{{2 + n_lvef}-{1 + n_lvef + n_sex + 1}}}"

combined_tabular = (
    f"   \\begin{{tabular}}{{{col_spec_all}}}\n"
    "      \\toprule\n"
    f"{header_top}\n"
    f"{cmidrule_lvef}\n"
    f"{cmidrule_sex}\n"
    f"{header_mid}\n"
    "      \\midrule\n"
    + "\n".join(rows) + "\n"
    "      \\bottomrule\n"
    "   \\end{tabular}\n"
)

combined_tabular = combined_tabular.replace("_", " ")

latex_combined = (
    "\\begin{table*}\n"
    "   \\centering\n"
    "   \\begin{threeparttable}\n"
    "   \\caption{Spearman rank correlation between parameters and ensemble prediction by LVEF subgroup and sex.}\n"
    "   \\label{tab:spearman_corr_lvef_sex}\n"
    f"{combined_tabular}"
    "   \\begin{tablenotes}\\footnotesize\n"
    "   \\item Higher absolute $\\rho$ indicates stronger monotonic association. "
    "The $p$ column reports two-sided permutation test $p$-values for the difference in correlation between males and females, "
    "based on \\num{10000} random reassignments of sex labels. "
    "Values are capped at $<\\num{0.001}$.\n"
    "   \\end{tablenotes}\n"
    "   \\end{threeparttable}\n"
    "\\end{table*}\n"
)

with open("tables/spearman_table_lvef_sex_p.tex", "w") as f:
    f.write(latex_combined)

print(latex_combined)


In [ ]:
100-kept_filtered["sex"].mean()*100
kept_filtered["age"].mean()

In [ ]:
from scipy.stats import spearmanr, norm
import numpy as np
import pandas as pd


# ---------- helper to compute correlations for a given dataframe ----------
def compute_corr_for_df(df, param_cols, target_col="ensemble"):
    out = []
    for col in param_cols:
        if col in ("sex", "FOEDSELSNR"):
            continue
        corr, _ = spearmanr(df[col], df[target_col], nan_policy="omit")
        out.append((col, corr))
    return pd.DataFrame(out, columns=["parameter", "rho"])

# ---------- overall correlations (used for ordering + reporting if needed) ----------
overall_corr_df = compute_corr_for_df(kept_filtered, plot_cols)

# sort by |ρ| overall
overall_corr_df = (
    overall_corr_df
    .assign(abs_corr=lambda d: d["rho"].abs())
    .sort_values("abs_corr", ascending=False)
    .drop(columns=["abs_corr"])
)

# we'll keep the parameter order fixed from here on
param_order = overall_corr_df["parameter"].tolist()
pretty_names = overall_corr_df["parameter"].tolist()

# ---------- LVEF subgroups ----------
lvef_groups = {
    r"$\leq$ 40\%": kept_filtered["LVEF"] <= 40,
    r"41--49\%": (kept_filtered["LVEF"] > 40) & (kept_filtered["LVEF"] < 50),
    r"$\geq$ 50\%": kept_filtered["LVEF"] >= 50,
}
lvef_cols = list(lvef_groups.keys())

# ---------- formatting + stats helpers ----------
def fmt_rho(x):
    if pd.isna(x):
        return ""
    return f"{x:.2f}"

def fisher_z(r):
    r = np.clip(r, -0.999999, 0.999999)
    return np.arctanh(r)

def pval_diff_spearman(r1, n1, r2, n2):
    """Two-sided p-value for difference in independent correlations."""
    if np.isnan(r1) or np.isnan(r2) or n1 < 4 or n2 < 4:
        return np.nan
    z1, z2 = fisher_z(r1), fisher_z(r2)
    z_diff = (z1 - z2) / np.sqrt(1/(n1 - 3) + 1/(n2 - 3))
    p = 2 * (1 - norm.cdf(abs(z_diff)))
    return p


def perm_test_diff_spearman(df, l_mask, param, target="ensemble", B=1000, seed=123):
    rng = np.random.default_rng(seed)

    # subset to this LVEF group and needed columns
    sub = df.loc[l_mask, [param, target, "sex"]].dropna()

    # split by sex
    m = sub[sub["sex"] == 1.0]
    f = sub[sub["sex"] == 0.0]

    if len(m) < 4 or len(f) < 4:
        return np.nan

    # observed difference
    rho_m, _ = spearmanr(m[param], m[target])
    rho_f, _ = spearmanr(f[param], f[target])
    obs_diff = rho_m - rho_f

    diffs = np.empty(B)

    for b in range(B):
        # permute sex labels
        permuted = sub.copy()
        permuted["sex"] = rng.permutation(permuted["sex"].values)

        m_p = permuted[permuted["sex"] == 1.0]
        f_p = permuted[permuted["sex"] == 0.0]

        # if a permutation accidentally gives too few of one sex, skip
        if len(m_p) < 4 or len(f_p) < 4:
            diffs[b] = 0.0
            continue

        rho_m_p, _ = spearmanr(m_p[param], m_p[target])
        rho_f_p, _ = spearmanr(f_p[param], f_p[target])
        diffs[b] = rho_m_p - rho_f_p

    p = np.mean(np.abs(diffs) >= np.abs(obs_diff))
    return p


# we assume sex coded as 1.0 (male) and 0.0 (female)
sex_codes = [1.0, 0.0]
sex_labels = {1.0: "Male", 0.0: "Female"}

# sample sizes per (LVEF, sex)
n_by_lvef_sex = {}
for l_label, l_mask in lvef_groups.items():
    for s in sex_codes:
        mask = l_mask & (kept_filtered["sex"] == s)
        n_by_lvef_sex[(l_label, s)] = int(mask.sum())

# ===== collect rhos and p-values =====
row_data = []
all_abs_rhos = []

for pname, param in zip(pretty_names, param_order):
    rhos = {}   # (lvef_label, sex_code) -> rho
    pvals = {}  # lvef_label -> p (male vs female in that LVEF)
    for l_label, l_mask in lvef_groups.items():
        # correlations within each sex for this LVEF group
        for s in sex_codes:
            df_sub = kept_filtered.loc[l_mask & (kept_filtered["sex"] == s)]
            rho, _ = spearmanr(df_sub[param], df_sub["ensemble"], nan_policy="omit")
            rhos[(l_label, s)] = rho
            if not np.isnan(rho):
                all_abs_rhos.append(abs(rho))

        # p-value comparing male vs female in this LVEF group
        r_m = rhos[(l_label, 1.0)]
        r_f = rhos[(l_label, 0.0)]
        n_m = n_by_lvef_sex[(l_label, 1.0)]
        n_f = n_by_lvef_sex[(l_label, 0.0)]
        # p = pval_diff_spearman(r_m, n_m, r_f, n_f)
        # p = pval_diff_spearman(r_m, n_m, r_f, n_f)
        df_m = kept_filtered.loc[l_mask & (kept_filtered["sex"] == 1.0)]
        df_f = kept_filtered.loc[l_mask & (kept_filtered["sex"] == 0.0)]
        p = perm_test_diff_spearman(kept_filtered, l_mask, param,B=N_BOOTSTRAP)

        pvals[l_label] = p

    row_data.append(
        {
            "pname": pname,
            "rhos": rhos,
            "pvals": pvals,
        }
    )

max_abs_rho = max(all_abs_rhos) if all_abs_rhos else 0.0

# map |rho| to green!xx with:
#   xx = 50 for max |rho|
#   xx = 0 for |rho| <= 0.4 * max |rho|
#   linearly in between otherwise
def fmt_rho_colored(rho, max_abs):
    if pd.isna(rho) or max_abs == 0:
        return ""
    abs_r = abs(rho)
    half = 0.4 * max_abs
    if abs_r <= half:
        level = 0
    elif abs_r >= max_abs:
        level = 50
    else:
        level = int(round(50 * (abs_r - half) / (max_abs - half)))
    # wrap the number in \num
    return f"\\cellcolor{{green!{level}}} \\num{{{rho:.2f}}}"


# ===== build LaTeX rows =====
rows = []
for rd in row_data:
    cells = []
    for l_label in lvef_cols:
        # Male, Female rho cells
        for s in sex_codes:
            rho = rd["rhos"][(l_label, s)]
            cells.append(fmt_rho_colored(rho, max_abs_rho))

        # p-value cell for this LVEF group
        pval = rd["pvals"][l_label]
        if pd.isna(pval):
            p_str = ""
        else:
            if pval < 0.001:
                # only the numeric part is wrapped in \num
                p_str = r"<\num{0.001}"
            else:
                p_str = rf"\num{{{pval:.3f}}}"
        cells.append(p_str)

    row = "      " + " & ".join([rd["pname"]] + cells) + r" \\"
    rows.append(row)

# ----- table layout -----
# For each LVEF: Male, Female, p  -> 3 columns
n_lvef = len(lvef_cols)
cols_per_lvef = 3
total_data_cols = n_lvef * cols_per_lvef

col_spec_all = "l" + "r" * total_data_cols

# top header: LVEF subgroups
header_top = (
    "      & "
    + " & ".join(
        [
            f"\\multicolumn{{{cols_per_lvef}}}{{c}}{{{label}}}"
            for label in lvef_cols
        ]
    )
    + r" \\"
)

# mid header: alternating sex and p under each LVEF
mid_cells = []
for _ in lvef_cols:
    mid_cells.extend(["Male", "Female", "p"])
header_mid = "      Parameter & " + " & ".join(mid_cells) + r" \\"

# cmidrules for each LVEF block
cmidrules = []
start = 2  # first data column
for i in range(n_lvef):
    end = start + cols_per_lvef - 1
    cmidrules.append(f"      \\cmidrule(lr){{{start}-{end}}}")
    start = end + 1
cmidrules_str = "\n".join(cmidrules)

combined_tabular = (
    f"   \\begin{{tabular}}{{{col_spec_all}}}\n"
    "      \\toprule\n"
    f"{header_top}\n"
    f"{cmidrules_str}\n"
    f"{header_mid}\n"
    "      \\midrule\n"
    + "\n".join(rows) + "\n"
    "      \\bottomrule\n"
    "   \\end{tabular}\n"
)

# avoid LaTeX issues with underscores in names
combined_tabular = combined_tabular.replace("_", " ")

latex_combined = (
    "\\begin{table*}\n"
    "   \\centering\n"
    "   \\begin{threeparttable}\n"
    "   \\caption{Spearman rank correlation between parameters and ensemble prediction by LVEF subgroup and sex.}\n"
    "   \\label{tab:spearman_corr_lvef_sex_6groups}\n"
    f"{combined_tabular}"
    "   \\begin{tablenotes}\\footnotesize\n"
    "   \\item Higher absolute $\\rho$ indicates stronger monotonic association. "
    "Within each LVEF subgroup, the $p$ columns report two-sided permutation test $p$-values for the difference in correlation between males and females.\n"
    "   \\end{tablenotes}\n"
    "   \\end{threeparttable}\n"
    "\\end{table*}\n"
)


with open("tables/spearman_table_lvef_sex_6groups.tex", "w") as f:
    f.write(latex_combined)

print(latex_combined)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ================== Minimal utilities ==================

def scott_bandwidth(x):
    x = np.asarray(x, float)
    n = max(len(x), 2)
    sd = np.std(x, ddof=1) if n > 1 else 1.0
    return 1.06 * sd * n ** (-1/5) * 2

def silverman_bandwidth(x):
    x = np.asarray(x, float)
    n = max(len(x), 2)
    sd = np.std(x, ddof=1) if n > 1 else 1.0
    iqr = np.subtract(*np.percentile(x, [75, 25])) if n > 1 else 1.0
    sigma = min(sd, iqr / 1.34) if iqr > 0 else sd
    return 0.9 * sigma * n ** (-1/5)

def kernel_mean_std(x, y, x_grid):
    """Gaussian Nadaraya–Watson for E[y|x] and Std[y|x] on a fixed grid."""
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    h = silverman_bandwidth(x)
    U = (x[:, None] - x_grid[None, :]) / h
    w = np.exp(-0.5 * U**2)

    wsum = w.sum(axis=0)
    mu = (w * y[:, None]).sum(axis=0) / wsum
    # plt.plot(mu)
    # plt.show()
    m2 = (w * (y[:, None]**2)).sum(axis=0) / wsum
    std = np.sqrt(np.maximum(m2 - mu**2, 0.0))
    return mu, std

def _weighted_quantile(v, w, qs):
    idx = np.argsort(v)
    v_sorted = v[idx]
    w_sorted = w[idx]
    cw = np.cumsum(w_sorted)
    if cw[-1] <= 0:
        return np.full(len(qs), np.nan)
    cw /= cw[-1]
    return np.interp(qs, cw, v_sorted)

def rank_uniform(x):
    x = np.asarray(x, float)
    n = x.size
    u = (pd.Series(x).rank(method="average").to_numpy() - 0.5) / n
    eps = 1e-6
    return np.clip(u, eps, 1 - eps)

def inv_ecdf(u, x):
    u = np.asarray(u, float)
    return np.quantile(x, u, method="linear")

# ================== Rank-x smoothers (logit always) ==================

def smooth_prob_curve_rankx(x, p, n_grid=200):
    """Smooth E[p|x] after rank-transforming ONLY x (logit smoothing)."""
    x = np.asarray(x, float)
    eps = 1e-5
    p = np.clip(np.asarray(p, float), eps, 1 - eps)
    u = rank_uniform(x)
    u_grid = np.linspace(0.001, 0.999, n_grid)

    z = np.log(p / (1 - p))  # logit
    muz, sdz = kernel_mean_std(u, z, x_grid=u_grid)
    sigmoid = lambda t: 1 / (1 + np.exp(-t))
    mean_p = sigmoid(muz)
    lo = sigmoid(muz - sdz)
    hi = sigmoid(muz + sdz)
    xg_real = inv_ecdf(u_grid, x)
    return xg_real, mean_p, lo, hi

def kernel_quantile_band_rankx(x, y, qs=(0.25, 0.75), n_grid=200):
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    u = rank_uniform(x)
    u_grid = np.linspace(0.001, 0.999, n_grid)
    h = silverman_bandwidth(u)
    U = (u[:, None] - u_grid[None, :]) / h
    W = np.exp(-0.5 * U**2)
    qlo = np.empty_like(u_grid, dtype=float)
    qhi = np.empty_like(u_grid, dtype=float)
    for j in range(u_grid.size):
        wj = W[:, j]
        if wj.sum() <= 0:
            qlo[j] = qhi[j] = np.nan
        else:
            qlo[j], qhi[j] = _weighted_quantile(y, wj, qs)
    xg_real = inv_ecdf(u_grid, x)
    return xg_real, qlo, qhi

# ================== SETTINGS ==================
N_GRID = 400
QS = (0.2, 0.8)

# ================== Plotting (expects `columns` and `kept`) ==================
fig2, axs2 = plt.subplots(3, 5, figsize=(15, 9))
# plot_cols = ['age', 'LVEF', 'GLS', 'E/e'', 'MAPSE', 'LAv', 'TAPSE', "E"]

legend_handles, legend_labels = None, None

for i, (ax, column) in enumerate(zip(axs2.flatten(), plot_cols)):
    sub = kept_filtered[kept_filtered[column].notna()]
    sub = sub.copy() # NOTE this is how to find out HF or not!
    x = sub[column].values
    y = sub["ensemble"].values  # probabilities in [0,1]
    y_diag = sub["Heart failure"].values  # ground-truth labels (0/1)

    # num_positive_cases = np.sum(y_diag)
    # # let y be the thresholded ensemble predictions with same number of positive cases as y_diag
    # threshold = np.sort(sub["ensemble"].values)[-num_positive_cases] if num_positive_cases > 0 else 1.1
    # y = (sub["ensemble"].values >= threshold).astype(int)

    ind = np.argsort(x)
    x = x[ind]
    y = y[ind]
    y_diag = y_diag[ind]

    # display limits (1st–99th pct)
    lims = np.percentile(x, [0.5, 99.5])
    ax.set_xlim(lims)

    # Mean curve (logit) + quantile spread band (rank-x)
    # xg, mean_p, lo_p, hi_p = smooth_prob_curve_rankx(x, y, n_grid=N_GRID)
    # xg, mean_p_diag, lo_diag, hi_diag = smooth_prob_curve_rankx(x, y_diag, n_grid=N_GRID)

    qrs = 0.49
    xg_q, qlo, qhi = kernel_quantile_band_rankx(x, y, qs=(qrs, 1-qrs), n_grid=N_GRID)
    qm = (qlo + qhi) / 2
    ax.plot(xg_q, qm, linewidth=2.0, color="C0", zorder=2, label="Median")
    qrs = 0.25
    xg_q, qlo, qhi = kernel_quantile_band_rankx(x, y, qs=(qrs, 1-qrs), n_grid=N_GRID)
    ax.fill_between(xg_q, qlo, qhi, alpha=0.4, color="C0", zorder=1,
                    label=f"{int(qrs*100)}--{int((1-qrs)*100)}% range", edgecolor="none")

    ax.set_ylim(0, 1)
    ax.set_title(column.replace("_", " "))
    ax.set_xlabel("")
    if i in (0,5,10):
        # ax.set_ylabel("Model-predicted risk of HF (%)")
        ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0], ["0", "25", "50", "75", "100"])
    else:
        ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0], ["", "", "", "", ""])
    # ax.grid(axis='y')

    # # histogram axis behind the main one
    ax2 = ax.twinx()
    # make a kdeplot of density of x
    sns.kdeplot(x, ax=ax2, color='gray', fill=True, alpha=0.3, edgecolor='gray', linewidth=0.0, label="Parameter density", zorder=0, bw_adjust=0.5)
    # ax2.set_ylim(0, 1)

    # ax2.hist(x, bins=np.linspace(lims[0], lims[1], 18), density=True, alpha=1, color='#CCCCCC', zorder=0, rwidth=2.0, edgecolor='#CCCCCC')
    ax2.set_yticks([])
    ax2.patch.set_visible(False)      # remove white background
    ax.set_zorder(ax2.get_zorder() + 1)  # bring model curve to front
    ax.patch.set_visible(False)

    # remove spines of the histogram axis
    ax2.spines['top'].set_visible(False)
    ax2.spines['right'].set_visible(False)
    ax2.spines['bottom'].set_visible(False)
    ax2.spines['left'].set_visible(False)
    ax2.set_ylabel("")

    # ax as well (top and right)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    if i == 0:
        legend_handles, legend_labels = ax.get_legend_handles_labels()
        # add legend handles for ax2
        handles2, labels2 = ax2.get_legend_handles_labels()
        legend_handles += handles2
        legend_labels += labels2

# single legend outside the grid
if legend_handles:
    # remove the legend border
    fig2.legend(legend_handles, legend_labels, loc="upper center", ncol=3, frameon=False, fontsize=11)

# set y label for the entire figure
fig2.text(0.0, 0.5, "Model-predicted risk of HF (%)", va='center', rotation='vertical', fontsize=12)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig("figures/png/rankx_quartile_bands.png", dpi=300)
# plt.savefig("figures/pgf/rankx_quartile_bands.pgf")
plt.show()


In [ ]:
kept_filtered

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ================== SETTINGS ==================
N_GRID = 400
QS = (0.2, 0.8)

# ================== Plotting (expects `columns` and `kept_filtered`) ==================
fig2, axs2 = plt.subplots(3, 5, figsize=(15, 9))

# Define how to map gender values to labels and colors
sex_info = {
    0: {"label": "Female", "color": "C0"},
    1: {"label": "Male",   "color": "C1"},
}

legend_handles, legend_labels = None, None

for i, (ax, column) in enumerate(zip(axs2.flatten(), plot_cols)):
    sub_all = kept_filtered[kept_filtered[column].notna()].copy()

    # display limits (1st–99th pct) based on the full sample
    x_all = sub_all[column].to_numpy()
    lims = np.percentile(x_all, [0.5, 99.5])
    ax.set_xlim(lims)
    ax.set_ylim(0, 1)
    ax.set_title(column.replace("_", " "))
    ax.set_xlabel("")

    if i in (0, 5, 10):
        ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0], ["0", "25", "50", "75", "100"])
    else:
        ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0], ["", "", "", "", ""])

    # --- Background density (all patients, not sex-split) ---
    ax2 = ax.twinx()
    sns.kdeplot(
        x_all,
        ax=ax2,
        color="gray",
        fill=True,
        alpha=0.3,
        edgecolor="gray",
        linewidth=0.0,
        label="Parameter density",
        zorder=0,
        bw_adjust=0.5,
    )
    ax2.set_yticks([])
    ax2.patch.set_visible(False)
    ax.set_zorder(ax2.get_zorder() + 1)
    ax.patch.set_visible(False)

    # remove spines of the histogram axis
    for spine in ("top", "right", "bottom", "left"):
        ax2.spines[spine].set_visible(False)
    # remove top/right spines of main axis
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    # --- Sex-specific curves ---
    for sex_val, info in sex_info.items():
        sub = sub_all[sub_all["sex"] == sex_val]
        if sub.empty:
            continue

        x = sub[column].to_numpy()
        y = sub["ensemble"].to_numpy()  # probabilities in [0,1]
        y_diag = sub["Heart failure"].to_numpy()  # ground-truth labels (0/1), unused here

        # sort by x
        ind = np.argsort(x)
        x = x[ind]
        y = y[ind]
        y_diag = y_diag[ind]

        # 98% central band around median
        qrs = 0.49
        xg_q, qlo, qhi = kernel_quantile_band_rankx(
            x, y, qs=(qrs, 1 - qrs), n_grid=N_GRID
        )
        qm = (qlo + qhi) / 2.0
        # median curve
        ax.plot(
            xg_q,
            qm,
            linewidth=2.0,
            color=info["color"],
            zorder=2,
            label=f"{info['label']} median",
        )

        # 50% central band
        qrs = 0.25
        xg_q, qlo, qhi = kernel_quantile_band_rankx(
            x, y, qs=(qrs, 1 - qrs), n_grid=N_GRID
        )
        ax.fill_between(
            xg_q,
            qlo,
            qhi,
            alpha=0.25,
            color=info["color"],
            zorder=1,
            edgecolor="none",
            label=f"{info['label']} {int(qrs*100)}–{int((1-qrs)*100)}% range",
        )

    # collect legend info only once
    if i == 0:
        legend_handles, legend_labels = ax.get_legend_handles_labels()
        handles2, labels2 = ax2.get_legend_handles_labels()
        legend_handles += handles2
        legend_labels += labels2

# single legend outside the grid
if legend_handles:
    fig2.legend(
        legend_handles,
        legend_labels,
        loc="upper center",
        ncol=5,
        frameon=False,
        fontsize=11,
    )

# set y label for the entire figure
fig2.text(
    0.0,
    0.5,
    "Model-predicted risk of HF (%)",
    va="center",
    rotation="vertical",
    fontsize=12,
)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig("figures/png/rankx_quartile_bands_sex_split.png", dpi=300)
plt.show()

In [ ]:
kept["FOEDSELSNR"].nunique()

In [ ]:
# for each parameter in plot_cols, split by sex and show the distribution of each parameter
fig, axs = plt.subplots(3, 5, figsize=(15, 9))
for ax, column in zip(axs.flatten(), plot_cols):
    sub = kept_filtered[kept_filtered[column].notna()]
    sub = sub.query("LVEF > 50")
    males = sub[sub['sex'] == 1.0]
    females = sub[sub['sex'] == 0.0]
    sns.kdeplot(males[column], ax=ax, alpha=0.5, label='Male', color='blue', fill=True, edgecolor='blue', linewidth=0.0)
    sns.kdeplot(females[column], ax=ax, alpha=0.5, label='Female', color='red', fill=True, edgecolor='red', linewidth=0.0)
    ax.set_title(column)
    ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
keep_cols = ["ensemble"] + plot_cols
drop_cols = []#["LVEDVOL", "A", "Cardiac_output", "PWT", "sex", "E/e'", "CI", "LAvume", "age"]
for col in drop_cols:
    if col in keep_cols:
        keep_cols.remove(col)

corr = np.abs(kept[keep_cols].corr(method='pearson'))
plt.figure(figsize=(10,8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlation Matrix")
plt.show()

corr_spearman = (kept[keep_cols].corr(method='spearman')) * 100
# set the diagonal to nan
# np.fill_diagonal(corr_spearman.values, np.nan)
# set the top part to nan as well

from matplotlib.colors import LinearSegmentedColormap

# Create a green-only diverging colormap (white at center)
green_cmap = LinearSegmentedColormap.from_list(
    "green_diverging",
    ["#009900", "#FFFFFF", "#009900"],  # dark green → white → dark green
    N=256
)

# corr_spearman = corr_spearman.where(np.tril(np.ones(corr_spearman.shape), k=0).astype(bool))
plt.figure(figsize=(10,10))
sns.heatmap(corr_spearman, annot=True, fmt=".0f", cmap=green_cmap, vmin=-80, vmax=80, cbar=False)
plt.title("Spearman Rank Correlation (\%)")
# tilt xticks
plt.xticks(rotation=35, ha='right')

# for both x and y ticks, replace underscores with spaces
plt.yticks(rotation=0)
plt.yticks(ticks=np.array(list(range(len(corr_spearman.index))))+0.5, labels=[label.replace("_", " ") for label in corr_spearman.index])
plt.xticks(ticks=np.array(list(range(len(corr_spearman.columns))))+0.5, labels=[label.replace("_", " ") for label in corr_spearman.columns])
plt.savefig("figures/pgf/spearman_corr_matrix.pgf")
plt.show()



In [ ]:
# def calculate_num_abnormal(row):
#     """
#     Add one abnormal count if: average e' > 7 
#     E/e' < 15
#     TR velocity < 2.8 m/s
#     LA volume index < 34 ml/m²
#     """
#     return sum([
#         row["e'"]*100 < 7,
#         row["E/e'"] > 15,
#         row["PASP"] > 39,
#         row["LAVi"] > 34,
#     ])
# kept["num_abnormal"] = kept.apply(calculate_num_abnormal, axis=1)
# print(kept["num_abnormal"].value_counts().sort_index())



# kept_subdf = kept.copy().query("(num_abnormal != 2) & (LVEF > 50)")

# import seaborn as sns
# plt.figure(figsize=(8,6))
# sns.boxplot(x="num_abnormal", y="ensemble", data=kept_subdf, palette="Set2")
# plt.xlabel("Number of Abnormal Echocardiographic Parameters")
# plt.ylabel("Model-predicted Risk of Heart Failure")
# plt.title("Risk of Heart Failure vs. Number of Abnormal Parameters")
# plt.ylim(0, 1)
# plt.grid(axis='y')
# plt.show()

In [ ]:
import pandas as pd
import numpy as np
from textwrap import dedent
import re

# ---- helpers ----
def wrap_num_in_num(s: str):
    """Wrap numbers in \\num{} for siunitx; strip thousands separators inside \\num."""
    # if not isinstance(s, str) or s.strip() == "":
    #     return s
    def repl(m):
        digits = m.group(1).replace(",", "")
        return f"\\num{{{digits}}}"
    return re.sub(r"(?<![A-Za-z0-9])([0-9][0-9,\.]*)", repl, s)

def fmt_n_pct(n, d):
    """Return 'n (p\\%)' string with escaped percent sign."""
    if d == 0 or pd.isna(d):
        return "0 (0\\%)"
    pct = 100.0 * n / d
    return f"{n:,} ({pct:.0f}\\%)"

def fmt_mean_sd(a):
    a = pd.to_numeric(pd.Series(a), errors="coerce").dropna()
    if a.empty: return ""
    return f"{a.mean():.1f} ({a.std(ddof=1):.1f})"

def fmt_median_iqr(a):
    a = pd.to_numeric(pd.Series(a), errors="coerce").dropna()
    if a.empty: return ""
    q1, q2, q3 = np.percentile(a, [25, 50, 75])
    return f"{q2:.1f} [{q1:.1f}, {q3:.1f}]"

def normalize_sex(x):
    if pd.isna(x): return np.nan
    s = str(x).strip().lower()
    if s in {"f", "female", "kvinne", "k"}: return "female"
    if s in {"m", "male", "mann", "gutt"}: return "male"
    return np.nan

_REGEX_CACHE = {}
def any_match_regex(code_list, regex_list):
    if not isinstance(code_list, (list, tuple)): return False
    for pat in regex_list:
        if pat not in _REGEX_CACHE:
            _REGEX_CACHE[pat] = re.compile(pat)
        rx = _REGEX_CACHE[pat]
        for c in code_list:
            if isinstance(c, str) and rx.match(c):
                return True
    return False

# ---- single-table builder using only subdiag + IS_BASELINE ----
def build_baseline_from_subdiag(subdiag: pd.DataFrame, n_total: int = None) -> pd.DataFrame:
    expected = ['PERSONID','OMSORGSEPISODEID','KODE','TEKST','DIAGNOSETID',
                'HOVEDDIAGNOSE','FOEDSELSNR','time','age','sex','IS_BASELINE']
    missing = set(expected) - set(subdiag.columns)
    if missing:
        raise ValueError(f"subdiag missing columns: {missing}")

    d = subdiag.copy()
    d["KODE"] = d["KODE"].astype(str).str.upper().str.strip()
    d["DIAGNOSETID"] = pd.to_datetime(d["DIAGNOSETID"], errors="coerce")
    d["time"] = pd.to_datetime(d["time"], errors="coerce")
    d["IS_BASELINE"] = pd.to_numeric(d["IS_BASELINE"], errors="coerce").fillna(0).astype(int)
    d = d.dropna(subset=["time"])

    # earliest echo per patient -> index row (for age/sex)
    idx_rows = (d.sort_values(["PERSONID","time"])
                  .drop_duplicates(subset=["PERSONID"], keep="first")
                  .loc[:, ["PERSONID","time","age","sex"]]
                  .rename(columns={"time":"index_time"}))
    idx_rows["age"] = pd.to_numeric(idx_rows["age"], errors="coerce")
    idx_rows["Sex_norm"] = idx_rows["sex"].apply(normalize_sex)

    cohort_ids = idx_rows["PERSONID"].unique()
    N = n_total or len(cohort_ids)

    # collect codes at/before index (IS_BASELINE) and all-time
    codes_idx = (d.loc[d["IS_BASELINE"] == 1]
                   .groupby("PERSONID")["KODE"].apply(list)
                   .reindex(cohort_ids, fill_value=[]))
    codes_ever = (d.groupby("PERSONID")["KODE"].apply(list)
                   .reindex(cohort_ids, fill_value=[]))

    # assemble rows
    rows = []
    rows.append(("Patients, n", "", f"{N:,}", ""))

    if idx_rows["age"].notna().any():
        rows.append(("Age", "", "", ""))
        rows.append(("\\quad Years, mean (SD)", "", fmt_mean_sd(idx_rows["age"]), ""))
        rows.append(("\\quad Years, median [IQR]", "", fmt_median_iqr(idx_rows["age"]), ""))

    s = idx_rows["Sex_norm"]
    rows.append(("Sex", "", "", ""))
    rows.append(("\\quad Female", "", fmt_n_pct(int((s == "female").sum()), N), ""))
    rows.append(("\\quad Male",   "", fmt_n_pct(int((s == "male").sum()), N), ""))
    miss = int(s.isna().sum())
    if miss > 0:
        rows.append(("\\quad Missing", "", fmt_n_pct(miss, N), ""))

    rows.append(("Comorbidities", "", "", ""))
    comorbid_sorted = []
    for name, regex_list in COMORBIDITY_REGEX.items():
        n_idx = int(codes_idx.apply(lambda cs: any_match_regex(cs, regex_list)).sum())
        n_evr = int(codes_ever.apply(lambda cs: any_match_regex(cs, regex_list)).sum())
        icd_label = ", ".join(COMORBIDITY_MAP.get(name, []))
        comorbid_sorted.append((name, icd_label, n_idx, n_evr))
    comorbid_sorted.sort(key=lambda x: x[2], reverse=True)
    for name, icd_label, n_idx, n_evr in comorbid_sorted:
        rows.append((f"\\quad {name}", icd_label, fmt_n_pct(n_idx, N), fmt_n_pct(n_evr, N)))

    return pd.DataFrame(rows, columns=["Characteristic", "ICD-10", "On/before index", "All time"])

# ---- build + LaTeX ----
baseline_df = build_baseline_from_subdiag(subdiag, n_total=len(kept_filtered))

caption = "Baseline characteristics for the full echocardiography cohort: age, sex, and comorbidities recorded before the index date and across all available time."
label = "tab:baseline_full_echo"

latex = dedent(f"""
\\begin{{table*}}[!htbp]
\\centering
\\caption{{{caption}}}
\\label{{{label}}}
\\begin{{threeparttable}}
\\begin{{tabular}}{{@{{}}l l rr@{{}}}}
\\toprule
Characteristic & ICD-10 & Before index & All time \\\\
\\midrule
""").lstrip()

for ch, icd, v1, v2 in baseline_df.itertuples(index=False):
    latex += f"{ch} & {wrap_num_in_num(icd)} & {wrap_num_in_num(v1)} & {wrap_num_in_num(v2)} \\\\\n"

latex += dedent(r"""
\bottomrule
\end{tabular}
\begin{tablenotes}[para]
\footnotesize
Before index includes diagnostic codes registered before the first ECG; All time includes diagnoses recorded after the index.
\end{tablenotes}
\end{threeparttable}
\end{table*}
""")

with open("tables/baseline_table_full_echo.tex", "w") as f:
    # drop empty rows from latex and indent all but first and last row
    new_latex = []
    for row in latex.splitlines():
        if row.strip() == "":
            continue
        if row.startswith("\\begin{table}") or row.startswith("\\end{table}"):
            new_latex.append(row)
        else:
            new_latex.append("\t" + row)
    new_latex[-1] = new_latex[-1].lstrip()
    latex = "\n".join(new_latex)
    f.write(latex)

print(latex)



In [ ]:
subdiag

subdiag["hf_before_ecg"] = subdiag["KODE"].str.startswith(("I50", "I42", "I110")) & subdiag["time"].ge(subdiag["DIAGNOSETID"])
subdiag["hf_after_ecg"] = subdiag["KODE"].str.startswith(("I50", "I42", "I110")) & subdiag["time"].lt(subdiag["DIAGNOSETID"])

# create a mapping dict from PERSONID to either 0, 1 or 2.
# 0 if never hf, 1 if hf after ecg (but not before), and 2 if  before ecg

hf_status = {}
for pid, group in subdiag.groupby("FOEDSELSNR"):
    has_hf_before = group["hf_before_ecg"].any()
    has_hf_after = group["hf_after_ecg"].any()
    if has_hf_before:
        hf_status[pid] = 2
    elif has_hf_after:
        hf_status[pid] = 1
    else:
        hf_status[pid] = 0

# apply this to kept
kept["hf_status"] = kept["FOEDSELSNR"].map(hf_status).fillna(0).astype(int)
kept["hf_status"].value_counts()

In [ ]:
subdiag["PERSONID"].nunique()

In [ ]:
fnr_map_path = "../../../data/hf/mapping_foedselsnr.csv"
fnr_map = pd.read_csv(fnr_map_path)
# echo = echo.merge(fnr_map[["FOEDSELSNR_SINGLE", "FOEDSELSNR_DOUBLE"]], left_on="ID", right_on="FOEDSELSNR_SINGLE", how="left")
# echo = echo[~echo["FOEDSELSNR_DOUBLE"].isna()]
# echo = echo.rename(columns={"FOEDSELSNR_DOUBLE": "FOEDSELSNR"})
# echo = echo.drop(columns=["FOEDSELSNR_SINGLE"])
# echo = echo[~echo["FOEDSELSNR"].isna()]
# echo = echo[["SeriesDate", "FOEDSELSNR"]]
# echo = echo.drop_duplicates(subset=["SeriesDate", "FOEDSELSNR"], keep="first")

In [ ]:
blood = pd.read_csv("/home/stenheli/projects/heart-failure-phenotyping/src/scripts/cache_lung/blodprov_cache.csv")
demo = pd.read_csv("/dataset/ecg/tabular/raw/2025_32_Demorgafi.csv")
demo.head()

# demo foeds
demo_foedselsnr = demo[["FOEDSELSNR"]].drop_duplicates().copy()
echo_foedselsnrs = echo[["FOEDSELSNR"]].drop_duplicates().copy()
# intersect
common_foedselsnr = pd.merge(demo_foedselsnr, echo_foedselsnrs, on="FOEDSELSNR", how="inner")
print(f"Common FOEDSELSNR count: {len(common_foedselsnr)}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ================== Rank-x smoothers (logit always) ==================
# Key change: allow using a reference ECDF (x_ref) and a shared u_grid.

def smooth_prob_curve_rankx(x, p, u_grid=None, x_ref=None, n_grid=200):
    """
    Smooth E[p|x] after rank-transforming ONLY x (logit smoothing).
    The curve is placed at xg_real = inv_ecdf(u_grid, x_ref) if x_ref is given,
    otherwise at inv_ecdf(u_grid, x) (original behavior).
    """
    x = np.asarray(x, float)
    eps = 1e-5
    p = np.clip(np.asarray(p, float), eps, 1 - eps)
    u = rank_uniform(x)
    if u_grid is None:
        u_grid = np.linspace(0.01, 0.99, n_grid)

    z = np.log(p / (1 - p))  # logit
    muz, sdz = kernel_mean_std(u, z, x_grid=u_grid)
    sigmoid = lambda t: 1 / (1 + np.exp(-t))
    mean_p = sigmoid(muz)
    lo = sigmoid(muz - sdz)
    hi = sigmoid(muz + sdz)

    xg_real = inv_ecdf(u_grid, x_ref if x_ref is not None else x)
    return xg_real, mean_p, lo, hi

def kernel_quantile_band_rankx(x, y, qs=(0.25, 0.75), u_grid=None, x_ref=None, n_grid=200):
    """
    Kernel-smoothed conditional quantiles (rank-x). Plots at
    inv_ecdf(u_grid, x_ref) if x_ref provided; else at inv_ecdf(u_grid, x).
    """
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    u = rank_uniform(x)
    if u_grid is None:
        u_grid = np.linspace(0.01, 0.99, n_grid)
    h = scott_bandwidth(u)
    U = (u[:, None] - u_grid[None, :]) / h
    W = np.exp(-0.5 * U**2)
    qlo = np.empty_like(u_grid, dtype=float)
    qhi = np.empty_like(u_grid, dtype=float)
    for j in range(u_grid.size):
        wj = W[:, j]
        if wj.sum() <= 0:
            qlo[j] = qhi[j] = np.nan
        else:
            qlo[j], qhi[j] = _weighted_quantile(y, wj, qs)
    xg_real = inv_ecdf(u_grid, x_ref if x_ref is not None else x)
    return xg_real, qlo, qhi

# ================== SETTINGS ==================
N_GRID = 100
QS = (0.25, 0.75)

# ================== Plotting (FULL-group 1–99% support as reference) ==================
# Expects DataFrame `kept` with columns including the features, "ensemble" (prob), and "Heart failure" (0/1)
fig2, axs2 = plt.subplots(3, 5, figsize=(15, 8))
# plot_cols = ['age', 'LVEF', 'GLS', 'E/e'', 'TR_velocity', 'LAv', 'TAPSE', "E"]

grp_info = {
    0: {"label": "No HF", "color": "C0"},
    1: {"label": "HF after ECG",    "color": "C1"},
    2: {"label": "HF before ECG", "color": "C2"},
}

legend_handles, legend_labels = None, None

for i, (ax, column) in enumerate(zip(axs2.flatten(), plot_cols)):
    sub_all = kept[kept[column].notna()].copy()
    # sub_all = sub_all[sub_all["LVEF"]>50]
    if sub_all.empty:
        ax.set_visible(False)
        continue

    # Common u-grid and FULL-group reference ECDF (1–99%)
    ref_low, ref_high = 0.01, 0.99
    u_grid = np.linspace(ref_low, ref_high, N_GRID)
    x_all = sub_all[column].values
    x_ref = x_all  # <- FULL group reference for both HF strata
    lims = np.percentile(x_ref, [ref_low*100, ref_high*100])
    ax.set_xlim(lims)


    # Style the twin axis so it stays in the background
    ax2.set_yticks([])
    ax2.patch.set_visible(False)
    ax.set_zorder(ax2.get_zorder() + 1)
    ax.patch.set_visible(False)
    for sp in ['top', 'right', 'bottom', 'left']:
        ax2.spines[sp].set_visible(False)

    for sp in ['top', 'right', 'bottom', 'left']:
        ax2.spines[sp].set_visible(False)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    # Two stratified curves, both placed on FULL-group x grid
    for hf_val in (0, 1, 2):
        sub = sub_all[(sub_all["hf_status"] == hf_val)].copy()
        if sub.empty:
            continue

        x = sub[column].values
        y = sub["ensemble"].values  # probabilities in [0,1]

        xg, mean_p, _, _ = smooth_prob_curve_rankx(x, y, u_grid=u_grid, x_ref=x_ref, n_grid=N_GRID)
        xg_q, qlo, qhi = kernel_quantile_band_rankx(x, y, qs=QS, u_grid=u_grid, x_ref=x_ref, n_grid=N_GRID)

        ln, = ax.plot(
            xg, mean_p,
            linewidth=1.5,
            color=grp_info[hf_val]["color"],
            zorder=2,
            label=f"{grp_info[hf_val]['label']} mean",
        )
        ax.fill_between(
            xg_q, qlo, qhi,
            alpha=0.20,
            color=grp_info[hf_val]["color"],
            zorder=1,
            edgecolor='none',
            label=f"{grp_info[hf_val]['label']} {int(QS[0]*100)}–{int(QS[1]*100)}% spread",
        )

    ax.set_ylim(0, 1)
    ax.set_title(column.replace("_", " "))
    ax.set_xlabel("")
    if i in (0, 4):
        ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0], ["0", "25", "50", "75", "100"])
    else:
        ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0], ["", "", "", "", ""])
    ax.grid(axis='y')

    if i == 0:
        legend_handles, legend_labels = ax.get_legend_handles_labels()

# Single, de-duplicated legend
if legend_handles:
    uniq = {}
    for h, l in zip(legend_handles, legend_labels):
        if l not in uniq:
            uniq[l] = h
    fig2.legend(
        list(uniq.values()), list(uniq.keys()),
        loc="upper center", ncol=3, frameon=False, fontsize=11
    )
# Figure-level y label and layout
fig2.text(0.0, 0.5, "Model-predicted risk of HF (%)", va='center', rotation='vertical', fontsize=12)
plt.tight_layout(rect=[0, 0, 1, 0.9])

plt.show()